# Critical Thinking in Argumentative Writing (early access)

This evaluator rates a student argumentative essay for **Critical Thinking** on a five-level scale (Not Evident, Exploring, Analyzing, Integrating, Extending) across five indicators, plus an **Overall** rating.

* **synthesizing_sources** (2.1) — synthesize multiple sources (rated only for multi-source prompts)
* **evidence_strength** (2.2) — evaluate the strength of evidence
* **counterarguments** (3.1) — address counterarguments
* **facts_over_opinions** (3.2) — rely on facts over opinions
* **drawing_conclusions** (4.1) — draw specific conclusions (deductive reasoning)

The structured output lists each indicator with its verbatim **evidence**, a **rationale**, and its **rating** — produced in that order so the reasoning conditions the rating — and finally the **overall** rating, formed as the median of the indicators.

In [ ]:
%pip install -qU langchain-anthropic langchain pydantic typing_extensions python-dotenv

In [ ]:
import getpass
import os
import json
import hashlib
from pathlib import Path
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
import pprint as pp

In [ ]:
# Anthropic API key
load_dotenv()
if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")

In [ ]:
ASSETS_DIR = Path(".")

with open(ASSETS_DIR / "config.json") as f:
    CONFIG = json.load(f)

# Standalone schema files (config.json references them via $ref by path).
with open(ASSETS_DIR / "input_schema.json") as f:
    INPUT_SCHEMA = json.load(f)
with open(ASSETS_DIR / "output_schema.json") as f:
    OUTPUT_SCHEMA = json.load(f)

# Load every prompt message from disk and verify its sha256 against config
# (drift tripwire). Prompts are never hardcoded in this notebook.
PROMPT_MESSAGES = []  # (role, text), in config order
for msg_spec in CONFIG["steps"][0]["prompt"]["messages"]:
    role = msg_spec["role"]
    text = (ASSETS_DIR / msg_spec["source_path"]).read_text()
    actual = hashlib.sha256(text.encode("utf-8")).hexdigest()
    assert actual == msg_spec["sha256"], (
        f"prompt drift for role={role!r} ({msg_spec['source_path']}): "
        f"declared {msg_spec['sha256'][:12]}..., actual {actual[:12]}..."
    )
    PROMPT_MESSAGES.append((role, text))

_STEP = CONFIG["steps"][0]
print(f"Loaded {CONFIG['evaluator']['id']} from {ASSETS_DIR.resolve()}")
print(f"  model:       {_STEP['model']['name']}")
print(f"  temperature: {_STEP['generation']['temperature']}")
for role, text in PROMPT_MESSAGES:
    sha = hashlib.sha256(text.encode("utf-8")).hexdigest()[:12]
    print(f"  {role:>6}  ({len(text):>5} chars, sha {sha})")

In [ ]:
def evaluate_critical_thinking(assignment_text: str, sources: str, essay_text: str):
    """Run the Critical Thinking evaluator once and return a full I/O trace.

    Model, prompts, and the output schema are all read from the config on disk.
    parser.kind == "structured_output" -> native structured-output enforcement.
    """
    llm = ChatAnthropic(
        model=_STEP["model"]["name"],
        temperature=_STEP["generation"]["temperature"],
    )
    structured_llm = llm.with_structured_output(OUTPUT_SCHEMA, include_raw=True)
    prompt_template = ChatPromptTemplate.from_messages(PROMPT_MESSAGES)

    try:
        inputs = {
            "assignment_text": assignment_text,
            "sources": sources,
            "essay_text": essay_text,
        }
        rendered = prompt_template.format_messages(**inputs)
        raw = structured_llm.invoke(rendered)
        if raw.get("parsing_error"):
            raise ValueError(f"structured output parsing failed: {raw['parsing_error']}")
        return {
            "rendered_prompt": [m.model_dump() for m in rendered],
            "raw_text": raw["raw"].content,
            "formatted_output": raw["parsed"],
            "usage": getattr(raw["raw"], "usage_metadata", None),
        }
    except Exception as e:
        return f"Error evaluating essay: {e}"

In [ ]:
sample = {
    "assignment_text": "Some cities are considering banning cars from their downtown cores. Using the sources, argue whether your town should limit cars downtown.",
    "sources": "### Source 1: A Car-Free Downtown\nSeveral European cities closed their centers to cars; air quality improved and foot traffic to shops rose.\n\n### Source 2: The Cost of Going Car-Free\nCritics note delivery businesses and people with disabilities can be harmed when car access is removed without alternatives.",
    "essay_text": "Our town should limit cars downtown. Source 1 says that when European cities closed their centers to cars, the air got cleaner and more people walked to shops. This shows fewer cars can help health and local business. Source 2 warns deliveries and people with disabilities could be hurt, so the town would need to plan for those groups.",
}

result = evaluate_critical_thinking(**sample)
pp.pprint(result["formatted_output"] if isinstance(result, dict) else result)

In [ ]:
# Sniff-test runner: load fixtures.json and compare predicted vs expected.
# Fixtures carry a partial 'expected' (here, the Overall rating). The free-text
# rationale is non-deterministic, so we compare the rating label only.
LEVELS = OUTPUT_SCHEMA["$defs"]["Level"]["enum"]
with open(ASSETS_DIR / CONFIG["fixtures"]["path"]) as f:
    fixtures = json.load(f)
_ALLOW_ADJ = bool(CONFIG.get("fixtures", {}).get("tolerance", {}).get("allow_adjacent_levels", False))

def outcome(pred, exp):
    if pred == exp:
        return "exact"
    if _ALLOW_ADJ and pred in LEVELS and exp in LEVELS and abs(LEVELS.index(pred) - LEVELS.index(exp)) == 1:
        return "adjacent"
    return "fail"

print(f"Loaded {len(fixtures)} fixtures\n")
print(f"{'ID':<40} {'STATUS':<9} {'PREDICTED':<12} EXPECTED")
print("-" * 82)
for fx in fixtures:
    exp = fx["expected"]["overall"]["rating"]
    out = evaluate_critical_thinking(**fx["input"])
    if isinstance(out, str):
        print(f"{fx['id']:<40} {'ERROR':<9} {'-':<12} {exp}   ({out})")
        continue
    pred = out["formatted_output"]["overall"]["rating"]
    print(f"{fx['id']:<40} {outcome(pred, exp):<9} {pred:<12} {exp}")